# Mangrove NDVI Recovery After Hurricane Melissa

This notebook assesses NDVI recovery trajectories for benefit-providing mangroves after Hurricane Melissa. It compares the pre-event baseline with months 1-2, months 3-4, and months 5-6 after landfall. Damage is defined using the established threshold: relative NDVI decline from the pre-event baseline of at least 10%, evaluated only where baseline NDVI is at least 0.20.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display
from rasterio.features import rasterize

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.linewidth": 0.6,
    }
)

## Paths And Constants

In [ ]:
BASE = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
PAPER3 = BASE / "dphil_paper_3"

OUT_DIR = PAPER3 / "results" / "threats" / "hurricane_melissa_damage" / "recovery" / "mangrove_ndvi_recovery_months1_6"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MANGROVE_PATCHES_PATH = PAPER3 / "inputs" / "forces_of_nature_mangroves" / "mangroves.shp"
MANGROVE_PATCH_TABLE_PATH = PAPER3 / "results" / "threats" / "hurricane_melissa_damage" / "mangrove_eads_hurricane_damage" / "mangrove_ead_hurricane_damage_patch_table.csv"

NDVI_WINDOW_PATHS = {
    "before": PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif",
    "months_1_2": PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif",
    "months_3_4": PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_months3to4_after_epsg3448_2025-12-30_to_2026-02-28.tif",
    "months_5_6": PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_months5to6_after_epsg3448_2026-03-01_to_2026-04-29.tif",
}
NDVI_WINDOW_LABELS = {
    "before": "Pre-event",
    "months_1_2": "Months 1-2",
    "months_3_4": "Months 3-4",
    "months_5_6": "Months 5-6",
}
NDVI_TIME_ORDER = ["before", "months_1_2", "months_3_4", "months_5_6"]

REL_BASELINE_MIN = 0.20
REL_DAMAGE_THRESHOLD = -0.10
FIGURE_DPI = 300

for input_path in [MANGROVE_PATCHES_PATH, MANGROVE_PATCH_TABLE_PATH, *NDVI_WINDOW_PATHS.values()]:
    if not input_path.exists():
        raise FileNotFoundError(input_path)

OUT_DIR

## Helper Functions

In [ ]:
def read_ndvi_windows(window_paths: dict[str, Path]) -> tuple[dict[str, np.ndarray], dict]:
    """Read NDVI rasters and check that the windows share one grid."""
    ndvi_arrays = {}
    reference_profile = None
    for window_name, window_path in window_paths.items():
        with rasterio.open(window_path) as source_raster:
            profile = source_raster.profile.copy()
            ndvi_array = source_raster.read(1).astype("float32")
        if reference_profile is None:
            reference_profile = profile
        else:
            alignment_checks = {
                "crs": profile["crs"] == reference_profile["crs"],
                "transform": profile["transform"] == reference_profile["transform"],
                "height": profile["height"] == reference_profile["height"],
                "width": profile["width"] == reference_profile["width"],
            }
            if not all(alignment_checks.values()):
                raise ValueError(f"Raster alignment mismatch for {window_path}: {alignment_checks}")
        ndvi_arrays[window_name] = ndvi_array
    return ndvi_arrays, reference_profile


def valid_ndvi(ndvi_array: np.ndarray) -> np.ndarray:
    """Return valid NDVI values on the expected -1 to 1 range."""
    return np.isfinite(ndvi_array) & (ndvi_array >= -1.0) & (ndvi_array <= 1.0)


def pct(numerator: float, denominator: float) -> float:
    """Return percentage, preserving NaN when the denominator is zero."""
    return float(numerator / denominator * 100) if denominator else np.nan


def safe_mean(values: np.ndarray) -> float:
    """Return a finite mean or NaN for empty arrays."""
    return float(np.nanmean(values)) if values.size else np.nan


def save_figure(figure: plt.Figure, stem: str) -> list[Path]:
    """Save a figure to PNG, PDF, and SVG."""
    output_paths = []
    for suffix in ["png", "pdf", "svg"]:
        output_path = OUT_DIR / f"{stem}.{suffix}"
        figure.savefig(output_path, dpi=FIGURE_DPI, bbox_inches="tight", facecolor="white")
        output_paths.append(output_path)
    return output_paths


def summarize_recovery_group(
    group_name: str,
    group_mask: np.ndarray,
    ndvi_arrays: dict[str, np.ndarray],
    pixel_area_ha: float,
) -> dict:
    """Summarize NDVI change and recovery for one pixel group."""
    before_values = ndvi_arrays["before"][group_mask]
    months_1_2_values = ndvi_arrays["months_1_2"][group_mask]
    months_3_4_values = ndvi_arrays["months_3_4"][group_mask]
    months_5_6_values = ndvi_arrays["months_5_6"][group_mask]
    initial_drop = before_values - months_1_2_values
    positive_drop = initial_drop > 0

    recovery_fraction_3_4 = np.full(before_values.shape, np.nan, dtype="float64")
    recovery_fraction_5_6 = np.full(before_values.shape, np.nan, dtype="float64")
    recovery_fraction_3_4[positive_drop] = (months_3_4_values[positive_drop] - months_1_2_values[positive_drop]) / initial_drop[positive_drop]
    recovery_fraction_5_6[positive_drop] = (months_5_6_values[positive_drop] - months_1_2_values[positive_drop]) / initial_drop[positive_drop]

    pixel_count = int(group_mask.sum())
    return {
        "group": group_name,
        "pixel_count": pixel_count,
        "area_ha": pixel_count * pixel_area_ha,
        "mean_ndvi_before": safe_mean(before_values),
        "mean_ndvi_months_1_2": safe_mean(months_1_2_values),
        "mean_ndvi_months_3_4": safe_mean(months_3_4_values),
        "mean_ndvi_months_5_6": safe_mean(months_5_6_values),
        "mean_delta_months_1_2_minus_before": safe_mean(months_1_2_values - before_values),
        "mean_delta_months_3_4_minus_months_1_2": safe_mean(months_3_4_values - months_1_2_values),
        "mean_delta_months_5_6_minus_months_3_4": safe_mean(months_5_6_values - months_3_4_values),
        "mean_delta_months_5_6_minus_before": safe_mean(months_5_6_values - before_values),
        "mean_relative_change_months_1_2_pct": safe_mean((months_1_2_values - before_values) / before_values * 100),
        "mean_relative_change_months_3_4_pct": safe_mean((months_3_4_values - before_values) / before_values * 100),
        "mean_relative_change_months_5_6_pct": safe_mean((months_5_6_values - before_values) / before_values * 100),
        "pct_improved_months_3_4_vs_months_1_2": pct(np.sum(months_3_4_values > months_1_2_values), pixel_count),
        "pct_improved_months_5_6_vs_months_3_4": pct(np.sum(months_5_6_values > months_3_4_values), pixel_count),
        "pct_improved_months_5_6_vs_months_1_2": pct(np.sum(months_5_6_values > months_1_2_values), pixel_count),
        "pct_recovered_to_before_by_months_3_4": pct(np.sum(months_3_4_values >= before_values), pixel_count),
        "pct_recovered_to_before_by_months_5_6": pct(np.sum(months_5_6_values >= before_values), pixel_count),
        "mean_recovery_fraction_of_initial_drop_by_months_3_4_pct": safe_mean(recovery_fraction_3_4[np.isfinite(recovery_fraction_3_4)] * 100),
        "mean_recovery_fraction_of_initial_drop_by_months_5_6_pct": safe_mean(recovery_fraction_5_6[np.isfinite(recovery_fraction_5_6)] * 100),
        "mean_recovery_rate_ndvi_per_month_months_3_4_vs_months_1_2": safe_mean((months_3_4_values - months_1_2_values) / 2),
        "mean_recovery_rate_ndvi_per_month_months_5_6_vs_months_3_4": safe_mean((months_5_6_values - months_3_4_values) / 2),
        "mean_recovery_fraction_of_initial_drop_per_month_by_months_3_4_pct": safe_mean(recovery_fraction_3_4[np.isfinite(recovery_fraction_3_4)] * 100 / 2),
        "mean_recovery_fraction_of_initial_drop_per_month_by_months_5_6_pct": safe_mean(recovery_fraction_5_6[np.isfinite(recovery_fraction_5_6)] * 100 / 4),
    }


def summarize_patch(
    mangrove_id: int,
    patch_mask: np.ndarray,
    full_series_mask: np.ndarray,
    damaged_full_series_mask: np.ndarray,
    ndvi_arrays: dict[str, np.ndarray],
    pixel_area_ha: float,
) -> dict:
    """Summarize recovery for one mangrove patch."""
    patch_full_series_mask = patch_mask & full_series_mask
    patch_damaged_mask = patch_mask & damaged_full_series_mask
    full_series_summary = summarize_recovery_group("patch_full_series", patch_full_series_mask, ndvi_arrays, pixel_area_ha)
    damaged_summary = summarize_recovery_group("patch_damaged_months_1_2", patch_damaged_mask, ndvi_arrays, pixel_area_ha)
    return {
        "Mangrove_ID": mangrove_id,
        "full_series_pixel_count": full_series_summary["pixel_count"],
        "full_series_area_ha": full_series_summary["area_ha"],
        "damaged_months_1_2_pixel_count": damaged_summary["pixel_count"],
        "damaged_months_1_2_area_ha": damaged_summary["area_ha"],
        "pct_full_series_area_damaged_months_1_2": pct(damaged_summary["area_ha"], full_series_summary["area_ha"]),
        "mean_ndvi_before": full_series_summary["mean_ndvi_before"],
        "mean_ndvi_months_1_2": full_series_summary["mean_ndvi_months_1_2"],
        "mean_ndvi_months_3_4": full_series_summary["mean_ndvi_months_3_4"],
        "mean_ndvi_months_5_6": full_series_summary["mean_ndvi_months_5_6"],
        "damaged_mean_ndvi_before": damaged_summary["mean_ndvi_before"],
        "damaged_mean_ndvi_months_1_2": damaged_summary["mean_ndvi_months_1_2"],
        "damaged_mean_ndvi_months_3_4": damaged_summary["mean_ndvi_months_3_4"],
        "damaged_mean_ndvi_months_5_6": damaged_summary["mean_ndvi_months_5_6"],
        "damaged_pct_recovered_to_before_by_months_3_4": damaged_summary["pct_recovered_to_before_by_months_3_4"],
        "damaged_pct_recovered_to_before_by_months_5_6": damaged_summary["pct_recovered_to_before_by_months_5_6"],
        "damaged_mean_recovery_fraction_of_initial_drop_by_months_3_4_pct": damaged_summary["mean_recovery_fraction_of_initial_drop_by_months_3_4_pct"],
        "damaged_mean_recovery_fraction_of_initial_drop_by_months_5_6_pct": damaged_summary["mean_recovery_fraction_of_initial_drop_by_months_5_6_pct"],
    }

## Load NDVI Windows And Benefit-Providing Mangrove Patches

In [ ]:
ndvi_arrays, ndvi_profile = read_ndvi_windows(NDVI_WINDOW_PATHS)
ndvi_transform = ndvi_profile["transform"]
ndvi_shape = (ndvi_profile["height"], ndvi_profile["width"])
pixel_area_ha = abs(ndvi_transform.a * ndvi_transform.e) / 10_000

mangrove_patches = gpd.read_file(MANGROVE_PATCHES_PATH).to_crs(ndvi_profile["crs"])
mangrove_patch_table = pd.read_csv(MANGROVE_PATCH_TABLE_PATH)
positive_mangrove_ids = set(
    mangrove_patch_table.loc[
        mangrove_patch_table["positive_avoided_ead_either"].fillna(False).astype(bool),
        "Mangrove_ID",
    ].astype(int)
)

mangrove_patches["Mangrove_ID"] = mangrove_patches["ID"].astype(int)
benefit_mangrove_patches = mangrove_patches[mangrove_patches["Mangrove_ID"].isin(positive_mangrove_ids)].copy()
patch_raster_shapes = [
    (geometry, int(mangrove_id))
    for geometry, mangrove_id in zip(benefit_mangrove_patches.geometry, benefit_mangrove_patches["Mangrove_ID"], strict=True)
    if geometry is not None and not geometry.is_empty
]
benefit_mangrove_id_array = rasterize(
    patch_raster_shapes,
    out_shape=ndvi_shape,
    transform=ndvi_transform,
    fill=0,
    dtype="int32",
    all_touched=False,
)
benefit_mangrove_mask = benefit_mangrove_id_array > 0

print(f"Benefit-providing mangrove patches: {len(benefit_mangrove_patches):,}")
print(f"Rasterized benefit-providing mangrove area: {benefit_mangrove_mask.sum() * pixel_area_ha:,.1f} ha")
print(f"Pixel area: {pixel_area_ha:.3f} ha")

## Classify Month 1-2 Damage And Full Recovery Series

In [ ]:
valid_masks = {window_name: valid_ndvi(ndvi_array) for window_name, ndvi_array in ndvi_arrays.items()}
baseline_eligible_mask = benefit_mangrove_mask & valid_masks["before"] & (ndvi_arrays["before"] >= REL_BASELINE_MIN)
full_series_mask = baseline_eligible_mask.copy()
for window_name in ["months_1_2", "months_3_4", "months_5_6"]:
    full_series_mask &= valid_masks[window_name]

months_1_2_valid_mask = baseline_eligible_mask & valid_masks["months_1_2"]
relative_change_months_1_2 = np.full(ndvi_shape, np.nan, dtype="float32")
np.divide(
    ndvi_arrays["months_1_2"] - ndvi_arrays["before"],
    ndvi_arrays["before"],
    out=relative_change_months_1_2,
    where=months_1_2_valid_mask,
)
damaged_months_1_2_mask = months_1_2_valid_mask & (relative_change_months_1_2 <= REL_DAMAGE_THRESHOLD)
damaged_full_series_mask = damaged_months_1_2_mask & valid_masks["months_3_4"] & valid_masks["months_5_6"]

coverage_summary = pd.DataFrame(
    [
        {"metric": "benefit_mangrove_rasterized_area_ha", "value": benefit_mangrove_mask.sum() * pixel_area_ha},
        {"metric": "baseline_eligible_area_ha", "value": baseline_eligible_mask.sum() * pixel_area_ha},
        {"metric": "full_series_area_ha", "value": full_series_mask.sum() * pixel_area_ha},
        {"metric": "damaged_months_1_2_area_ha", "value": damaged_months_1_2_mask.sum() * pixel_area_ha},
        {"metric": "damaged_months_1_2_full_series_area_ha", "value": damaged_full_series_mask.sum() * pixel_area_ha},
        {"metric": "pct_full_series_area_damaged_months_1_2", "value": pct(damaged_full_series_mask.sum(), full_series_mask.sum())},
    ]
)
coverage_summary

## Pixel-Level Recovery Summaries

In [ ]:
pixel_recovery_summary = pd.DataFrame(
    [
        summarize_recovery_group(
            "all_benefit_mangrove_pixels_with_full_series",
            full_series_mask,
            ndvi_arrays,
            pixel_area_ha,
        ),
        summarize_recovery_group(
            "damaged_months_1_2_benefit_mangrove_pixels_with_full_series",
            damaged_full_series_mask,
            ndvi_arrays,
            pixel_area_ha,
        ),
    ]
)
pixel_recovery_summary

## Patch-Level Recovery Summaries

In [ ]:
patch_recovery_rows = []
for mangrove_id in sorted(positive_mangrove_ids):
    patch_recovery_rows.append(
        summarize_patch(
            mangrove_id,
            benefit_mangrove_id_array == mangrove_id,
            full_series_mask,
            damaged_full_series_mask,
            ndvi_arrays,
            pixel_area_ha,
        )
    )

patch_recovery_summary = pd.DataFrame(patch_recovery_rows)
patch_context_columns = [
    "Mangrove_ID",
    "Parish",
    "TYPE",
    "area_ha",
    "positive_avoided_ead_usd_min",
    "positive_avoided_ead_usd_max",
    "positive_avoided_ead_either",
    "distance_to_track_km",
    "max_wind_threshold_intersected",
]
patch_recovery_summary = mangrove_patch_table[patch_context_columns].merge(
    patch_recovery_summary,
    on="Mangrove_ID",
    how="right",
)
patch_recovery_summary.head()

## Time-Window Summaries

In [ ]:
time_window_rows = []
for group_name, group_mask in [
    ("all_benefit_mangrove_pixels_with_full_series", full_series_mask),
    ("damaged_months_1_2_benefit_mangrove_pixels_with_full_series", damaged_full_series_mask),
]:
    before_values = ndvi_arrays["before"][group_mask]
    for window_name in NDVI_TIME_ORDER:
        window_values = ndvi_arrays[window_name][group_mask]
        time_window_rows.append(
            {
                "group": group_name,
                "window": window_name,
                "window_label": NDVI_WINDOW_LABELS[window_name],
                "pixel_count": int(group_mask.sum()),
                "area_ha": group_mask.sum() * pixel_area_ha,
                "mean_ndvi": safe_mean(window_values),
                "mean_delta_vs_before": safe_mean(window_values - before_values),
                "mean_relative_change_vs_before_pct": safe_mean((window_values - before_values) / before_values * 100),
            }
        )
time_window_summary = pd.DataFrame(time_window_rows)
time_window_summary

## Figures

In [ ]:
fig, axis = plt.subplots(figsize=(92 / 25.4, 70 / 25.4), dpi=FIGURE_DPI)
legend_labels = {
    "all_benefit_mangrove_pixels_with_full_series": "All benefit pixels",
    "damaged_months_1_2_benefit_mangrove_pixels_with_full_series": "Damaged in months 1-2",
}
for group_name, color in [
    ("all_benefit_mangrove_pixels_with_full_series", "#006d77"),
    ("damaged_months_1_2_benefit_mangrove_pixels_with_full_series", "#d73027"),
]:
    plot_rows = time_window_summary[time_window_summary["group"].eq(group_name)].set_index("window").loc[NDVI_TIME_ORDER]
    axis.plot(
        plot_rows["window_label"],
        plot_rows["mean_ndvi"],
        marker="o",
        linewidth=1.2,
        color=color,
        label=legend_labels[group_name],
    )
axis.axhline(REL_BASELINE_MIN, color="#7f7f7f", linewidth=0.7, linestyle="--", label="Baseline eligibility threshold")
axis.set_ylabel("Mean NDVI")
axis.set_title("Benefit-providing mangrove NDVI trajectory")
axis.grid(axis="y", linewidth=0.35, alpha=0.35)
axis.spines[["top", "right"]].set_visible(False)
axis.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.24), ncol=2)
fig.tight_layout()
mean_ndvi_figure_paths = save_figure(fig, "mangrove_mean_ndvi_recovery_months1_6")
display(fig)
plt.close(fig)

fig, axis = plt.subplots(figsize=(92 / 25.4, 70 / 25.4), dpi=FIGURE_DPI)
damaged_row = pixel_recovery_summary[
    pixel_recovery_summary["group"].eq("damaged_months_1_2_benefit_mangrove_pixels_with_full_series")
].iloc[0]
bar_labels = ["Recovered by\nmonths 3-4", "Recovered by\nmonths 5-6", "Improved\n3-4 vs 1-2", "Improved\n5-6 vs 3-4"]
bar_values = [
    damaged_row["pct_recovered_to_before_by_months_3_4"],
    damaged_row["pct_recovered_to_before_by_months_5_6"],
    damaged_row["pct_improved_months_3_4_vs_months_1_2"],
    damaged_row["pct_improved_months_5_6_vs_months_3_4"],
]
axis.bar(bar_labels, bar_values, color=["#1a9850", "#006837", "#74add1", "#4575b4"], width=0.62)
axis.set_ylim(0, 100)
axis.set_ylabel("Damaged benefit pixels (%)")
axis.set_title("Recovery indicators for mangrove pixels damaged in months 1-2")
axis.grid(axis="y", linewidth=0.35, alpha=0.35)
axis.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
recovery_indicator_figure_paths = save_figure(fig, "mangrove_recovery_indicators_months1_6")
display(fig)
plt.close(fig)

mean_ndvi_figure_paths + recovery_indicator_figure_paths

## Export Tables And Metadata

In [ ]:
coverage_summary_path = OUT_DIR / "mangrove_recovery_coverage_summary.csv"
pixel_recovery_summary_path = OUT_DIR / "mangrove_recovery_pixel_summary.csv"
patch_recovery_summary_path = OUT_DIR / "mangrove_recovery_patch_summary.csv"
time_window_summary_path = OUT_DIR / "mangrove_recovery_time_window_summary.csv"
metadata_path = OUT_DIR / "mangrove_recovery_metadata.csv"

metadata = pd.DataFrame(
    [
        {"name": "mangrove_patches_path", "value": str(MANGROVE_PATCHES_PATH)},
        {"name": "mangrove_patch_table_path", "value": str(MANGROVE_PATCH_TABLE_PATH)},
        {"name": "ndvi_before_path", "value": str(NDVI_WINDOW_PATHS["before"])},
        {"name": "ndvi_months_1_2_path", "value": str(NDVI_WINDOW_PATHS["months_1_2"])},
        {"name": "ndvi_months_3_4_path", "value": str(NDVI_WINDOW_PATHS["months_3_4"])},
        {"name": "ndvi_months_5_6_path", "value": str(NDVI_WINDOW_PATHS["months_5_6"])},
        {"name": "baseline_eligibility_threshold", "value": str(REL_BASELINE_MIN)},
        {"name": "damage_threshold", "value": "relative NDVI change months 1-2 vs before <= -0.10"},
        {"name": "pixel_area_ha", "value": str(pixel_area_ha)},
        {"name": "benefit_area_definition", "value": "mangrove patches with positive avoided EAD in either minimum or maximum coastal-flood scenario"},
    ]
)

coverage_summary.to_csv(coverage_summary_path, index=False)
pixel_recovery_summary.to_csv(pixel_recovery_summary_path, index=False)
patch_recovery_summary.to_csv(patch_recovery_summary_path, index=False)
time_window_summary.to_csv(time_window_summary_path, index=False)
metadata.to_csv(metadata_path, index=False)

[
    coverage_summary_path,
    pixel_recovery_summary_path,
    patch_recovery_summary_path,
    time_window_summary_path,
    metadata_path,
]

## Key Results

In [ ]:
display(coverage_summary)
display(pixel_recovery_summary)
display(time_window_summary)